# MiniGrid：跨 seed 的 WM target loss 与 policy 分布

本 notebook 完全自包含，只读取已有 CSV；不会导入或生成任何 `.py` 文件，也不会启动训练。图像只在 notebook 中显示，不保存 PNG。

In [ ]:
%matplotlib inline
from pathlib import Path
import csv
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# NeurIPS-style, paper-ready defaults (the plot is still displayed inline).
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.linewidth': 0.7,
    'lines.linewidth': 1.1,
    'figure.dpi': 140,
})

DR_REL = Path('outputs/results/dr/dr_summary_minigrid_mask5_focal_reservoir.csv')
P2E_REL = Path('outputs/results/p2e/p2e_baseline_minigrid_mask5_n7000_m1_summary.csv')
TARGET_BASELINE_REL = Path('outputs/results/target_baseline/target_baseline_minigrid_mask5.csv')
POLICY_REL = Path('outputs/results/dr_policy_target_eval/dr_policy_target_summary.csv')
REQUIRED_RESULTS = (DR_REL, P2E_REL, TARGET_BASELINE_REL, POLICY_REL)
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if all((p / rel).is_file() for rel in REQUIRED_RESULTS)), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        '找不到当前 workspace 的结果 CSV。请从 Curriculum_world_model_learning 根目录打开 notebook。'
    )
DR_LOSS_CSV = REPO_ROOT / DR_REL
P2E_LOSS_CSV = REPO_ROOT / P2E_REL
TARGET_BASELINE_LOSS_CSV = REPO_ROOT / TARGET_BASELINE_REL
POLICY_SUMMARY_CSV = REPO_ROOT / POLICY_REL
print('Repository:', REPO_ROOT)
print('DR CSV:', DR_LOSS_CSV)
print('P2E CSV:', P2E_LOSS_CSV)
print('Target baseline CSV:', TARGET_BASELINE_LOSS_CSV)
print('Policy summary CSV:', POLICY_SUMMARY_CSV)

## 1. WM target validation loss：DR、Target Random 与 P2E（mean ± std）

横轴统一为累计训练 transitions：DR 累加 `New_Data_Size`，Target Random 使用 `cumulative_data_size`，P2E 使用 `transitions`。

In [ ]:
baseline_specs = [
    {
        'name': 'DR baseline',
        'path': DR_LOSS_CSV,
        'seed_col': 'Seed',
        'loss_col': 'target_val_avg_val_loss_wm',
        'data_col': 'New_Data_Size',
        'data_is_increment': True,
        'color': '#2F6DAE',
    },
    {
        'name': 'Target Random',
        'path': TARGET_BASELINE_LOSS_CSV,
        'seed_col': 'Seed',
        'loss_col': 'target_val_avg_val_loss_wm',
        'data_col': 'cumulative_data_size',
        'data_is_increment': False,
        'color': '#D95F02',
    },
    {
        'name': 'Plan2Explore',
        'path': P2E_LOSS_CSV,
        'seed_col': 'seed',
        'loss_col': 'avg_target_loss',
        'data_col': 'transitions',
        'data_is_increment': False,
        'color': '#1B9E77',
    },
]

def load_seed_loss_curves(spec):
    with spec['path'].open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    seed_rows = defaultdict(list)
    for row in rows:
        seed = int(row[spec['seed_col']])
        iteration = int(row['Iter'])
        data_value = float(row[spec['data_col']])
        loss = float(row[spec['loss_col']])
        seed_rows[seed].append((iteration, data_value, loss))
    grouped = {}
    for seed, values in seed_rows.items():
        cumulative = 0.0
        grouped[seed] = {}
        for _, data_value, loss in sorted(values):
            if spec['data_is_increment']:
                cumulative += data_value
            else:
                cumulative = data_value
            grouped[seed][int(round(cumulative))] = loss
    if not grouped:
        raise ValueError(f"No rows found for {spec['name']}")
    common_data = sorted(set.intersection(*(set(v) for v in grouped.values())))
    if not common_data:
        raise ValueError(f"No common cumulative data points across seeds for {spec['name']}")
    matrix = np.asarray([[grouped[s][x] for x in common_data]
                         for s in sorted(grouped)], dtype=float)
    mean = matrix.mean(axis=0)
    std = matrix.std(axis=0, ddof=1) if matrix.shape[0] > 1 else np.zeros_like(mean)
    return common_data, matrix, mean, std

baseline_stats = {}
fig, ax = plt.subplots(figsize=(3.45, 2.55), dpi=140)
for spec in baseline_specs:
    common_data, matrix, mean, std = load_seed_loss_curves(spec)
    baseline_stats[spec['name']] = {
        'seed_count': int(matrix.shape[0]),
        'iteration_count': int(matrix.shape[1]),
        'last_cumulative_transitions': int(common_data[-1]),
        'first_mean_loss': float(mean[0]),
        'last_mean_loss': float(mean[-1]),
        'last_std_loss': float(std[-1]),
    }
    data_thousands = np.asarray(common_data, dtype=float) / 1000.0
    ax.plot(data_thousands, mean, color=spec['color'], linewidth=1.6,
            label=spec['name'])
    ax.fill_between(data_thousands, mean - std, mean + std,
                    color=spec['color'], alpha=0.16, linewidth=0,
                    label='_nolegend_')
ax.set_title('Target validation loss (mean ± std)', pad=3)
ax.set_xlabel(r'Cumulative transitions ($\times 10^3$)')
ax.set_ylabel('Target validation loss')
ax.grid(True, alpha=0.22, linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, handlelength=1.8, borderaxespad=0.2)
fig.tight_layout(pad=0.4)
plt.show()

In [ ]:
display({
    name: stats for name, stats in baseline_stats.items()
})

## 2. Policy 验证结果：seed 均值 ± 标准差

In [ ]:
with POLICY_SUMMARY_CSV.open(encoding='utf-8', newline='') as f:
    policy_rows = list(csv.DictReader(f))

policy_by_target = defaultdict(list)
for row in policy_rows:
    policy_by_target[row['target']].append({
        'seed': int(row['seed']),
        'mean_reward': float(row['mean_reward']),
        'success_rate': float(row['success_rate']),
    })
targets = sorted(policy_by_target)
x = np.arange(len(targets))
reward_mean = np.asarray([np.mean([r['mean_reward'] for r in policy_by_target[t]]) for t in targets])
reward_std = np.asarray([np.std([r['mean_reward'] for r in policy_by_target[t]], ddof=1)
                          if len(policy_by_target[t]) > 1 else 0.0 for t in targets])
success_mean = np.asarray([np.mean([r['success_rate'] for r in policy_by_target[t]]) for t in targets])
success_std = np.asarray([np.std([r['success_rate'] for r in policy_by_target[t]], ddof=1)
                           if len(policy_by_target[t]) > 1 else 0.0 for t in targets])
labels = [t.replace('target_task', 'target ') for t in targets]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
axes[0].bar(x, reward_mean, yerr=reward_std, capsize=5, color='#4C78A8', alpha=0.86)
axes[0].set_title('Policy dense reward across seeds')
axes[0].set_ylabel('Mean episode reward ± 1 std')
axes[1].bar(x, success_mean, yerr=success_std, capsize=5, color='#F58518', alpha=0.86)
axes[1].set_title('Policy success rate across seeds')
axes[1].set_ylabel('Success rate ± 1 std')
for ax in axes:
    ax.set_xticks(x, labels)
    ax.grid(axis='y', alpha=0.25)
for j, target in enumerate(targets):
    values = policy_by_target[target]
    jitter = np.linspace(-0.13, 0.13, len(values))
    axes[0].scatter(j + jitter, [v['mean_reward'] for v in values], color='#1f3b5b', s=16, zorder=3)
    axes[1].scatter(j + jitter, [v['success_rate'] for v in values], color='#8a4300', s=16, zorder=3)
plt.show()

In [ ]:
display([
    {
        'target': target,
        'seed_count': len(policy_by_target[target]),
        'mean_reward': float(reward_mean[i]),
        'std_reward': float(reward_std[i]),
        'success_rate': float(success_mean[i]),
        'std_success_rate': float(success_std[i]),
    }
    for i, target in enumerate(targets)
])